# Lab — Altair: Apple Mobility Data

In this lab, we are going to use Altair as a tool to investigate Apple Mobility data.

These anonymous, aggregate data [were published](https://covid19.apple.com/mobility) by Apple during the Covid-19 pandemic.  They show the relative use of Apple Maps for walking, driving, and transit directions, relative to January 13, 2020.  Days are dermined as midnight to midnight Pacific (or Apple Standard Time ;-) ).

In the original data, each day's value is in it's own column.  The `-transposed` version that we will work with has re-organized this data table to put each observation in its own row—a format that should be easier for us to work with.

In [1]:
## Import libraries

import pandas as pd
import altair as alt

# work-around to let Altair handle larger data sets
# This will use an external file to store our data instead of embedding it directly in the
# visualization
alt.data_transformers.enable('json')

pass # Don't show any output in this cell

In [102]:
# Choose a data set to work with.

# data = pd.read_csv('applemobilitytrends-2020-05-02-transposed.csv')
# data = pd.read_csv('applemobilitytrends-2020-06-10-transposed.csv')
#data = pd.read_csv('applemobilitytrends-2020-12-12-transposed.csv')
data = pd.read_csv('applemobilitytrends-2021-10-03-transposed.csv')

In [3]:
# Let's take a look at the data

data.head()

,Unnamed: 0,geo_type,region,transportation_type,alternative_name,sub-region,country,date,value
0,0,country/region,Albania,driving,NaN,NaN,NaN,2020-01-13,100.0
1,1,country/region,Albania,walking,NaN,NaN,NaN,2020-01-13,100.0
2,2,country/region,Argentina,driving,NaN,NaN,NaN,2020-01-13,100.0
3,3,country/region,Argentina,walking,NaN,NaN,NaN,2020-01-13,100.0
4,4,country/region,Australia,driving,AU,NaN,NaN,2020-01-13,100.0


In [4]:
# Now let's just look at Paris

paris = data[data['region'] == 'Paris']
paris

,Unnamed: 0,geo_type,region,transportation_type,alternative_name,sub-region,country,date,value
695,695,city,Paris,driving,NaN,Île-de-France Region,France,2020-01-13,100.00
696,696,city,Paris,transit,NaN,Île-de-France Region,France,2020-01-13,100.00
697,697,city,Paris,walking,NaN,Île-de-France Region,France,2020-01-13,100.00
5386,5386,city,Paris,driving,NaN,Île-de-France Region,France,2020-01-14,102.17
5387,5387,city,Paris,transit,NaN,Île-de-France Region,France,2020-01-14,100.82
...,...,...,...,...,...,...,...,...,...
2946644,2946644,city,Paris,transit,NaN,Île-de-France Region,France,2021-10-02,221.91
2946645,2946645,city,Paris,walking,NaN,Île-de-France Region,France,2021-10-02,131.96
2951334,2951334,city,Paris,driving,NaN,Île-de-France Region,France,2021-10-03,96.82
2951335,2951335,city,Paris,transit,NaN,Île-de-France Region,France,2021-10-03,179.23


Ok, now we are starting to get an idea of how the data is encoded.  Let's see what it looks like.  We're going to create a basic chart that encodes the date on the x axis and the value on the y axis.

In [5]:
alt.Chart(paris).mark_line().encode(
    x='date',
    y='value',
)

alt.Chart(...)

Hmmm, that doesn't look right.  What's going on here?

First, the date's aren't being interpreted as dates.  Can you remember how to tell Altair to treat a variable as temporal?

In [6]:
alt.Chart(paris).mark_line().encode(
    x='date:T', # <<
    y='value',
)

alt.Chart(...)

Ah, that's better, but it looks squashed now.  Let's explicitly give it a width.  Also, we want to distinguish between walking, transit, and driving directions, so let's color-code those.

In [7]:
alt.Chart(paris).mark_line().encode(
    x='date:T',
    y='value',
    color='transportation_type' # <<
).properties(width=800) # <<

alt.Chart(...)

Ah, that's looking better.  

It would probably be easier to follow if we had an explicit baseline.  Let's call out the baseline (100%).

In [8]:
# Where is the baseline?  Let's highlight it in the visualisation

base = alt.Chart(paris).mark_line().encode(
    x=alt.X('date:T', title=None),
    y=alt.Y('value', title="Relative Mobility"),
    color='transportation_type'
).properties(
    width=800
)

rule = base.mark_line(strokeWidth=1).encode(
    y='baseline:Q',
    color=alt.value('grey')
).transform_calculate(baseline='100')

rule + base

alt.LayerChart(...)

That's starting to tell a little more of a story.  We can see a sharp fall-off during the first confinement in March and the second in the end of October.

That said, there are some strange things going on.  For example, whateach of the curves seems to have a similar, regular, recurring jaggedness.  Is there some kind of periodicity going on?

One hypothesis might be that there's a weekday/weekend effect in the data.  Let's test that out.

In [9]:
# First we'll need to encode weekday/weekend-ness in our data.

from datetime import datetime

df = data
df['isWeekday'] = df.date.apply(lambda date: datetime.strptime(date, '%Y-%m-%d').weekday() < 5)

paris = data[data['region'] == 'Paris']

In [10]:
# Now let's distinguish weekends in the graph

weekends = alt.Chart(paris).mark_rect().encode(
    x=alt.X('date:O', axis=None),
    y=alt.YValue(1),
    color=alt.condition(alt.FieldEqualPredicate(True, 'isWeekday'), alt.value('transparent'), alt.value('lightgrey'))
)

(weekends + rule + base)

alt.LayerChart(...)

That's a little hard to read.  There are a bunch of ways we could try to address that.  Let's just try filtering down to the first quarter (trimestre en français) of 2020.

In [ ]:
# Look at just T1 2020 in Paris

paris_t1_2020 = paris[(paris['date'] >= '2020-01-01') & (paris['date'] < '2020-04-01')]

base = alt.Chart(paris_t1_2020).mark_line().encode(
    x=alt.X('date:O', title=None),
    y=alt.Y('value', title='Relative Mobility'),
    color=alt.Color('transportation_type', title='Transportation')
).properties(
    width=800
)

weekends = base.mark_rect().encode(
    x=alt.X('date:O', axis=None),
    y=alt.YValue(1),
    color=alt.condition(alt.FieldEqualPredicate(True, 'isWeekday'), alt.value('transparent'), alt.value('lightgrey'))
).properties(width=800, height=400)

weekends + base

This time that's pretty clear: there definitely seems to be a weekend effect that explains the periodicity.

# Exploring Cities

Now let's take a look at all the cities in the data.  A useful technique for this type of analysis is to use a faceted visualization, where in this case we'll facet the data on the city name.  That's basically a fancy way of saying that we're going to draw a separate chart for each city.

In [ ]:
cities = data[data.geo_type == 'city'] # Extract just the cities from the data

# Filter out a few cities that seem to break for some reason.
cities = cities[~cities.region.isin(['St. Louis', 'Tampa', 
                                     'Belo Horizonte', 'Madison'])]

base = alt.Chart(cities).mark_line(strokeWidth=1).encode(
    x='date:O',
    y=alt.Y('value', title='Relative Mobility'),
    color=alt.Color('transportation_type', title='Transportation')
)

# Since we're gonna have a lot of graphs, lets put them in 6 columns of 80×80 pixel graphs.
base.properties(width=80, height=80).facet('region', columns=6, spacing=40, bounds='flush', title='All modes, All cities')

This exercise, comes from a [blog post by Kieran Healey]() where he used R for his analyses.  Let's try to mimic the styles used in his analyses.  That's also a lot of cities, so let's also just choose a couple of cities to focus on.

In [13]:
focus_city_names = 'Amsterdam, Atlanta, Barcelona, Berlin, ' \
                   'Helsinki, London, New York City, Paris'.split(', ')

focus_cities = cities[cities.region.isin(focus_city_names)]

colorFields = ['driving', 'transit', 'walking']
colorValues = ['orange', 'blue', 'black']
transitColors = alt.Scale(domain=colorFields, range=colorValues)

base = alt.Chart(focus_cities).mark_line(strokeWidth=1).encode(
    x='date:O',
    y=alt.Y('value', title='Relative Mobility'),
    color=alt.Color('transportation_type', title='Transportation', scale=transitColors)
)

weekend_color = 'rgba(236, 246, 242, 0.6)'
weekends = alt.Chart(focus_cities).mark_rect().encode(
    x=alt.X('date:O', axis=None),
    y=alt.YValue(1),
    color=alt.condition(alt.FieldEqualPredicate(True, 'isWeekday'), alt.value('transparent'), alt.value(weekend_color))
)

rule = alt.Chart(focus_cities).mark_line(strokeWidth=1).encode(
    x='date:O',
    y='baseline:Q'
).transform_calculate(
    baseline='100'
)

(weekends + base + rule).properties(width=400, height=150) \
    .facet('region', columns=2, spacing=dict(row=50, column=40), bounds='flush', title='All modes, Selecty cities') \
    .configure_header(title=None)

alt.FacetChart(...)

# Open exercises

1. Try annotating the max day, the confinement periods, President Macron's speeches, etc.
2. Take a look at faceted chart of all the cities.  Are there any that look interesting to you for some reason?
    1. Choose two “weird” cities and construct a collection of graphs that help illustrate what seems weird.
    2. Try to figure out what might be going on.
    3. Create a visual explanation that supports your analysis and your conclusions, using a combination of markdown cells and graphs (similar to what we did to investigate the weekend effect).

## 2. Add annotations

In [ ]:
# event research was done with LLM
covid_events = {
    "Paris": {
        "first_covid_death": "2020-02-14",  # Chinese tourist at Lariboisière hospital
        "lockdown_1_start":  "2020-03-17",
        "lockdown_1_end":    "2020-05-11",
        "lockdown_2_start":  "2020-10-30",
        "lockdown_2_end":    "2020-12-15",  # Replaced by nationwide 9pm curfew
        "macron_speech_1":   "2020-03-12",  # First national address on COVID
        "macron_speech_2":   "2020-03-16",  # "Nous sommes en guerre" — announced lockdown
        "macron_speech_3":   "2020-10-28",  # Announced second lockdown
    },
    "London": {
        "first_covid_death": "2020-03-05",
        "lockdown_1_start":  "2020-03-23",
        "lockdown_1_end":    "2020-06-15",
        "lockdown_2_start":  "2020-11-05",
        "lockdown_2_end":    "2020-12-02",  # Returned to 3-tier regional system
        "macron_speech_1":   None,
        "macron_speech_2":   None,
        "macron_speech_3":   None,
    },
    "Berlin": {
        "first_covid_death": "2020-03-08",  # First death in Germany
        "lockdown_1_start":  "2020-03-22",  # Federal "Kontaktsperre" contact ban
        "lockdown_1_end":    "2020-04-19",
        "lockdown_2_start":  "2020-11-02",  # "Lockdown light" (bars/restaurants closed)
        "lockdown_2_end":    "2021-03-28",  # Gradual reopening of non-essential shops
        "macron_speech_1":   None,
        "macron_speech_2":   None,
        "macron_speech_3":   None,
    },
    "Amsterdam": {
        "first_covid_death": "2020-03-06",  # First death in the Netherlands
        "lockdown_1_start":  "2020-03-15",  # "Intelligent lockdown"
        "lockdown_1_end":    "2020-06-01",
        "lockdown_2_start":  "2020-12-15",  # Hard lockdown (all non-essential shops closed)
        "lockdown_2_end":    "2021-04-28",  # Curfew lifted, terraces reopened
        "macron_speech_1":   None,
        "macron_speech_2":   None,
        "macron_speech_3":   None,
    },
    "Barcelona": {
        "first_covid_death": "2020-02-13",  # First death in Spain (Valencia, diagnosed post-mortem)
        "lockdown_1_start":  "2020-03-15",  # National state of alarm
        "lockdown_1_end":    "2020-06-21",  # End of first state of alarm
        "lockdown_2_start":  "2020-10-25",  # Second state of alarm + nationwide night curfew
        "lockdown_2_end":    "2021-05-09",  # End of second state of alarm
        "macron_speech_1":   None,
        "macron_speech_2":   None,
        "macron_speech_3":   None,
    },
    "Helsinki": {
        "first_covid_death": "2020-03-21",  # First death in Finland (Helsinki/Uusimaa region)
        "lockdown_1_start":  "2020-03-16",  # State of emergency + Uusimaa border closure Mar 27
        "lockdown_1_end":    "2020-05-13",  # State of emergency lifted
        "lockdown_2_start":  "2021-03-01",  # Second state of emergency (third wave)
        "lockdown_2_end":    "2021-04-27",  # Second state of emergency ended
        "macron_speech_1":   None,
        "macron_speech_2":   None,
        "macron_speech_3":   None,
    },
    "New York City": {
        "first_covid_death": "2020-03-13",  # 82-year-old woman with underlying conditions
        "lockdown_1_start":  "2020-03-22",  # "NY on PAUSE" executive order
        "lockdown_1_end":    "2020-06-08",  # Phase 1 reopening in NYC
        "lockdown_2_start":  None,           # No formal second lockdown (micro-cluster zones only)
        "lockdown_2_end":    None,
        "macron_speech_1":   None,
        "macron_speech_2":   None,
        "macron_speech_3":   None,
    },
    "Atlanta": {
        "first_covid_death": "2020-03-12",  # First death in Georgia
        "lockdown_1_start":  "2020-04-03",  # Georgia statewide shelter-in-place
        "lockdown_1_end":    "2020-04-24",  # Governor Kemp began reopening Georgia
        "lockdown_2_start":  None,           # No formal second lockdown in Georgia
        "lockdown_2_end":    None,
        "macron_speech_1":   None,
        "macron_speech_2":   None,
        "macron_speech_3":   None,
    },
}

records = []
for city, events in covid_events.items():
    for event, date in events.items():
        if date is not None:
            records.append({"region": city, "date": date, "event": event})
events_df = pd.DataFrame(records)


In [63]:
yearly_events = {
    "new_year":     ["2020-01-01", "2021-01-01"],
    "spring_start": ["2020-03-20", "2021-03-20"],
    "summer_start": ["2020-06-20", "2021-06-21"],
    "autumn_start": ["2020-09-22", "2021-09-22"],
    "winter_start": ["2020-12-21", "2021-12-21"],
}

yearly_records = [
    {"region": city, "yearly_event": event, "date": date}
    for event, dates in yearly_events.items()
    for date in dates
    for city in focus_city_names
]

yearly_events_df = pd.DataFrame(yearly_records)

In [65]:
focus_city_names = 'Amsterdam, Atlanta, Barcelona, Berlin, ' \
                   'Helsinki, London, New York City, Paris'.split(', ')

focus_cities = cities[cities.region.isin(focus_city_names)]

colorFields = ['driving', 'transit', 'walking']
colorValues = ['orange', 'blue', 'black']
transitColors = alt.Scale(domain=colorFields, range=colorValues)

focus_cities_with_events = focus_cities.merge(
    yearly_events_df[['region', 'date', 'yearly_event']],
    on=['region', 'date'],
    how='left'
)

focus_cities_with_events = focus_cities_with_events.merge(
        events_df[['region', 'date', 'event']],
    on=['region', 'date'],
    how='left'
)


base = alt.Chart(focus_cities_with_events).mark_line(strokeWidth=1).encode(
    x='date:O',
    y=alt.Y('value', title='Relative Mobility'),
    color=alt.Color('transportation_type', title='Transportation', scale=transitColors)
)

weekend_color = 'rgba(236, 246, 242, 0.6)'
weekends = alt.Chart(focus_cities_with_events).mark_rect().encode(
    x=alt.X('date:O', axis=None),
    y=alt.YValue(1),
    color=alt.condition(alt.FieldEqualPredicate(True, 'isWeekday'), alt.value('transparent'), alt.value(weekend_color))
)

rule = alt.Chart(focus_cities).mark_line(strokeWidth=1).encode(
    x='date:O',
    y='baseline:Q'
).transform_calculate(
    baseline='100'
)

events_only = focus_cities_with_events.dropna(subset=["event"])

event_rules = (
    alt.Chart(focus_cities_with_events)
    .transform_filter("datum.event !== null")
    .mark_rule(color="red", strokeWidth=1, opacity=0.7)
    .encode(
        x="date:O",
        tooltip=["event:N", "date:O"],
    )
)


yearly_events_only = focus_cities_with_events.dropna(subset=["yearly_event"])

yearly_event_rules = (
    alt.Chart(focus_cities_with_events)
    .transform_filter("datum.yearly_event !== null")
    .mark_rule(color="steelblue", strokeWidth=1, opacity=0.7)
    .encode(
        x="date:O",
        tooltip=["event:N", "date:O"],
    )
)


(weekends + base + event_rules + rule).properties(width=400, height=150).facet('region', columns=2, spacing=dict(row=50, column=40), 
           bounds='flush', title='All modes, Select cities')\
    .configure_header(title=None)

alt.FacetChart(...)

In [169]:
def plot_specific_cities(focus_city_names , cities=cities ):
    focus_cities = cities[cities.region.isin(focus_city_names)]

    colorFields = ['driving', 'transit', 'walking']
    colorValues = ['orange', 'blue', 'black']
    transitColors = alt.Scale(domain=colorFields, range=colorValues)

    focus_cities_with_events = focus_cities.merge(
        yearly_events_df[['region', 'date', 'yearly_event']],
        on=['region', 'date'],
        how='left'
    )

    focus_cities_with_events = focus_cities_with_events.merge(
            events_df[['region', 'date', 'event']],
        on=['region', 'date'],
        how='left'
    )


    base = alt.Chart(focus_cities_with_events).mark_line(strokeWidth=1).encode(
        x='date:O',
        y=alt.Y('value', title='Relative Mobility'),
        color=alt.Color('transportation_type', title='Transportation', scale=transitColors)
    )

    weekend_color = 'rgba(236, 246, 242, 0.6)'
    weekends = alt.Chart(focus_cities_with_events).mark_rect().encode(
        x=alt.X('date:O', axis=None),
        y=alt.YValue(1),
        color=alt.condition(alt.FieldEqualPredicate(True, 'isWeekday'), alt.value('transparent'), alt.value(weekend_color))
    )

    rule = alt.Chart(focus_cities).mark_line(strokeWidth=1).encode(
        x='date:O',
        y='baseline:Q'
    ).transform_calculate(
        baseline='100'
    )

    events_only = focus_cities_with_events.dropna(subset=["event"])

    event_rules = (
        alt.Chart(focus_cities_with_events)
        .transform_filter("datum.event !== null")
        .mark_rule(color="red", strokeWidth=1, opacity=0.7)
        .encode(
            x="date:O",
            tooltip=["event:N", "date:O"],
        )
    )


    yearly_events_only = focus_cities_with_events.dropna(subset=["yearly_event"])

    yearly_event_rules = (
        alt.Chart(focus_cities_with_events)
        .transform_filter("datum.yearly_event !== null")
        .mark_rule(color="steelblue", strokeWidth=1, opacity=0.7)
        .encode(
            x="date:O",
            tooltip=["event:N", "date:O"],
        )
    )


    return (weekends + base + event_rules + rule + yearly_event_rules).properties(width=400, height=150)\
        .facet(facet=alt.Facet("region:N", sort=focus_city_names),
              columns=2, spacing=dict(row=30, column=50), 
            bounds='flush', title='All modes, Select cities')\
        .configure_header(title=None) \
        .resolve_scale(y="independent") 


## Weird City

Choice: Barcelona because transportation explodes and Atlanta because transit decreases a lot.

### Atlanta

In [ ]:
most_pop_us_cities_50 = 'New York City, Los Angeles, Chicago, Houston, Phoenix, Philadelphia, San Antonio, San Diego, Dallas, Fort Worth, Jacksonville, Austin, San Jose, Charlotte, Columbus, Indianapolis, San Francisco, Seattle, Denver, Nashville, Oklahoma City, Washington, El Paso, Las Vegas, Boston, Detroit, Louisville, Portland, Memphis, Baltimore, Milwaukee, Albuquerque, Fresno, Tucson, Sacramento, Atlanta, Kansas City, Mesa, Raleigh, Colorado, Springs, Miami, Omaha, Virginia, Beach, Long Beach, Oakland, Minneapolis, Bakersfield, Tulsa, Tampa, Aurora'.split(', ')

plot_specific_cities(most_pop_us_cities_50)

alt.FacetChart(...)

We see that this is not an effect link to Atlanta. It is the same in almost all large US cities. But we see that some cities recover earlier than other ones. The order of the plots above was ordered by population. Atlanta is known to have the busiest airport in the world, so maybe it is linked. Let's perform the same but for the ten more busy airport.

In [133]:
busiest_airports = 'Atlanta,Dallas,Chicago,Denver,Los Angeles,New York City,Orlando,Miami,Las Vegas,San Francisco - Bay Area,Charlotte,Seattle,Phoenix,Houston,Boston,Minneapolis,Detroit,New York City'.split(',')
plot_specific_cities(busiest_airports)

alt.FacetChart(...)

In [134]:
import altair as alt
import pandas as pd

# Top 20 busiest US airports (2023, passengers in millions) from wikipedia - LLM arranged in a dataframe
data_airport = pd.DataFrame([
    {"iata": "ATL", "name": "Atlanta (ATL)",       "lat": 33.6407, "lon": -84.4277,  "passengers": 106_302_208},
    {"iata": "DFW", "name": "Dallas/FW (DFW)",     "lat": 32.8998, "lon": -97.0403,  "passengers": 85_660_127},
    {"iata": "ORD", "name": "Chicago O'Hare (ORD)","lat": 41.9742, "lon": -87.9073,  "passengers": 84_851_825},
    {"iata": "DEN", "name": "Denver (DEN)",         "lat": 39.8561, "lon": -104.6737, "passengers": 82_427_962},
    {"iata": "LAX", "name": "Los Angeles (LAX)",    "lat": 33.9425, "lon": -118.4081, "passengers": 73_709_594},
    {"iata": "JFK", "name": "New York JFK (JFK)",   "lat": 40.6413, "lon": -73.7781,  "passengers": 62_629_455},
    {"iata": "MCO", "name": "Orlando (MCO)",         "lat": 28.4294, "lon": -81.3089,  "passengers": 57_675_573},
    {"iata": "MIA", "name": "Miami (MIA)",           "lat": 25.7959, "lon": -80.2870,  "passengers": 55_314_661},
    {"iata": "LAS", "name": "Las Vegas (LAS)",       "lat": 36.0840, "lon": -115.1537, "passengers": 54_989_185},
    {"iata": "SFO", "name": "San Francisco (SFO)",   "lat": 37.6213, "lon": -122.3790, "passengers": 54_532_613},
    {"iata": "CLT", "name": "Charlotte (CLT)",       "lat": 35.2140, "lon": -80.9431,  "passengers": 53_574_392},
    {"iata": "SEA", "name": "Seattle (SEA)",          "lat": 47.4502, "lon": -122.3088, "passengers": 52_715_181},
    {"iata": "PHX", "name": "Phoenix (PHX)",          "lat": 33.4373, "lon": -112.0078, "passengers": 51_620_420},
    {"iata": "IAH", "name": "Houston (IAH)",          "lat": 29.9902, "lon": -95.3368,  "passengers": 48_131_213},
    {"iata": "BOS", "name": "Boston (BOS)",           "lat": 42.3656, "lon": -71.0096,  "passengers": 43_236_013},
    {"iata": "MSP", "name": "Minneapolis (MSP)",      "lat": 44.8848, "lon": -93.2223,  "passengers": 36_071_627},
    {"iata": "DTW", "name": "Detroit (DTW)",          "lat": 42.2162, "lon": -83.3554,  "passengers": 33_372_682},
    {"iata": "LGA", "name": "LaGuardia (LGA)",        "lat": 40.7769, "lon": -73.8740,  "passengers": 32_791_050},
])



Compute the average transit per airport for the last 30 days of data that we have and plot it as a color scale on the map

In [157]:
data["date"] = pd.to_datetime(data["date"])
mean_rel_airport = []
for airport in busiest_airports:
    av = data.loc[
        (data["region"] == airport) &
        (data["transportation_type"] == "transit")
    ]["value"].tail(30).mean()
    print(airport, " : ",av)
    mean_rel_airport.append(av)
    
data_airport["mean_transit"] = mean_rel_airport


Atlanta  :  81.47733333333333
Dallas  :  85.507
Chicago  :  135.232
Denver  :  102.97333333333331
Los Angeles  :  101.38066666666666
New York City  :  127.41533333333334
Orlando  :  77.78533333333333
Miami  :  84.59933333333333
Las Vegas  :  113.56066666666668
San Francisco - Bay Area  :  107.50733333333334
Charlotte  :  94.02166666666668
Seattle  :  93.106
Phoenix  :  104.75066666666667
Houston  :  76.99733333333333
Boston  :  174.19966666666667
Minneapolis  :  95.43866666666668
Detroit  :  67.81
New York City  :  127.41533333333334


In [158]:
data_airport

,iata,name,lat,lon,passengers,mean_transit
0,ATL,Atlanta (ATL),33.6407,-84.4277,106302208,81.477333
1,DFW,Dallas/FW (DFW),32.8998,-97.0403,85660127,85.507000
2,ORD,Chicago O'Hare (ORD),41.9742,-87.9073,84851825,135.232000
3,DEN,Denver (DEN),39.8561,-104.6737,82427962,102.973333
4,LAX,Los Angeles (LAX),33.9425,-118.4081,73709594,101.380667
5,JFK,New York JFK (JFK),40.6413,-73.7781,62629455,127.415333
6,MCO,Orlando (MCO),28.4294,-81.3089,57675573,77.785333
7,MIA,Miami (MIA),25.7959,-80.2870,55314661,84.599333
8,LAS,Las Vegas (LAS),36.0840,-115.1537,54989185,113.560667
9,SFO,San Francisco (SFO),37.6213,-122.3790,54532613,107.507333


In [ ]:
# US states background - Done with a LLM
states = alt.topo_feature(
    "https://cdn.jsdelivr.net/npm/vega-datasets@v1.29.0/data/us-10m.json",
    "states"
)

background = alt.Chart(states).mark_geoshape(
    fill="lightgray",
    stroke="white",
    strokeWidth=0.5
).project("albersUsa")

points = alt.Chart(data_airport).mark_circle(
    opacity=0.7,
    color="steelblue",
    stroke="white",
    strokeWidth=0.8
).encode(
    longitude="lon:Q",
    latitude="lat:Q",
    size=alt.Size(
        "passengers:Q",
        scale=alt.Scale(range=[200, 3000]),
        legend=alt.Legend(title="Passengers (M)")
    ),
    color=alt.Color(
        "mean_transit:Q",
        scale=alt.Scale(scheme="viridis"),  # or "viridis", "redyellowgreen", etc.
        legend=alt.Legend(title="Avg relative mobility for transit")
    ),
    tooltip=[
        alt.Tooltip("name:N", title="Airport"),
        alt.Tooltip("passengers:Q", title="Passengers", format=","),
        alt.Tooltip("mean_transit:Q", title="Avg transit %", format=".1f"),
    ]).project("albersUsa")

chart = (background + points).properties(
    width=800,
    height=500,
    title="20 Busiest US Airports by Passengers (2023)"
)

chart.show()

alt.LayerChart(...)

The weirdness of the transit for Atlanta is not so exceptional when compared with other US large cities with a large airport. But this phenomenon seems to affect mainly city in the south-east of the US.

### Barcelona


In [164]:
mediteranean_cities= 'Barcelona,Madrid,Valencia,Sevilla,Marseille,Milan,Genoa,Rome,Venice,Lisbon'.split(',')
plot_specific_cities(mediteranean_cities)

alt.FacetChart(...)

This highly increased in transit is observed in many other cities in Spain or in the West Mediteranea. For a city like Marseille, we are sure that it is a tourism cycle as it goes up and down around summer. Madrid, as capital doesn't show this but Valencia and Barcelona do and they are known as being very touristic places.

We need more data to see if the increase is reduced during winter or not. We use this link: https://github.com/ActiveConclusion/COVID19_mobility/blob/master/apple_reports/apple_mobility_report.csv

In [174]:
data2 = pd.read_csv('apple_mobility_report.csv')
data2 = data2.melt(
    id_vars=["country", "sub-region", "subregion_and_city", "geo_type", "date"],
    value_vars=["driving", "transit", "walking"],
    var_name="transportation_type",
    value_name="value"
)
cities2 = data2[data2.geo_type == 'city'] # Extract just the cities from the data
cities2.rename(columns={"subregion_and_city": "region"}, inplace=True)


In [176]:
mediteranean_cities= 'Barcelona,Madrid,Valencia,Sevilla,Marseille,Rome'.split(',')
plot_specific_cities(mediteranean_cities, cities=cities2)

alt.FacetChart(...)

Just based on those data, it is difficult to say what is the reason. We can only see that Barcelona has a more similar evolution as Marseille and Valencia than Madrid and Rome, is it due to the proximity of the sea or the tourism? But if it was because of tourism, then a lot of people would be walking and driving around as well. It is also difficult to have the exact meaning of the category.